# Lab 1 — Ray Core Tasks & Actors (Fundamentals)

**Time:** ~15–20 min  
**Mode:** Complete by hand. No AI assistant.

## Learning objectives
By the end of this lab you should be able to:
1. Convert a sequential Python function into a Ray remote task.
2. Use `ray.get` and pass `ObjectRef`s between tasks (a small task graph).
3. Create and call a Ray actor that maintains state across many task invocations.

## Why this matters
These three primitives — tasks, ObjectRefs, and actors — underlie everything else you will build in Sessions 2–4 (Ray Data workers, Ray Serve replicas, Ray Train workers are all actors under the hood).

## Setup

In [ ]:
import ray, time, random

if not ray.is_initialized():
    ray.init()

## Exercise 1 — Parallelize a slow function

Below is a sequential function `score_product` that simulates a CPU-bound
scoring step (e.g., computing a quality score for a catalog row).

**Your task:**
1. Run the sequential version and note the wall time.
2. Make a Ray remote version of the same function (call it `score_product_remote`).
3. Launch 10 calls in parallel and retrieve the results with one `ray.get`.
4. Verify the parallel version is meaningfully faster.

**Acceptance criteria:** parallel wall time should be roughly 1 × `delay` plus small overhead (vs. ~10 × `delay` sequential).

In [ ]:
def score_product(product_id: int, delay: float = 0.5) -> float:
    time.sleep(delay)            # simulate CPU work
    return product_id * 0.1 + random.random()

# Sequential baseline — already complete
%time seq_results = [score_product(i) for i in range(10)]
print(seq_results[:3], '...')

In [ ]:
# TODO: define `score_product_remote` as a Ray task and run 10 calls in parallel.
# Use `%time` to compare wall time with the sequential version above.


## Exercise 2 — A tiny task graph

Build a 3-stage pipeline using Ray tasks. Each stage takes the previous
stage's `ObjectRef` as input (do NOT call `ray.get` between stages — Ray
will resolve the dependencies for you).

Stages:
1. `fetch(product_id)` → returns a dict `{"id": product_id, "raw_score": <float>}`
2. `tag(record)` → returns the same dict with an extra key `"tier"` set to
   `'gold'` if `raw_score > 0.5`, else `'silver'`.
3. `enrich(record)` → returns the same dict with a `"label"` key:
   `f"product-{record['id']}-{record['tier']}"`.

Build the graph for 5 product ids without calling `ray.get` until the end.

**Acceptance criteria:** `len(results) == 5` and every record has `id`, `raw_score`, `tier`, `label`.

In [ ]:
# TODO: implement fetch, tag, enrich as @ray.remote tasks and chain them.


## Exercise 3 — A stats actor

Define a `CatalogStats` actor that tracks a running count of products it has
seen, broken down by tier. It needs three methods:

- `record(tier: str)` — increments the counter for that tier.
- `totals()` — returns the dict of counters.
- `reset()` — zeros everything out.

Modify your `enrich` task from Exercise 2 to also call `stats.record.remote(tier)`.
Run the pipeline for 20 ids and confirm the totals add up.

**Acceptance criteria:** `sum(stats.totals().values()) == 20`.

In [ ]:
# TODO: define CatalogStats actor and wire it into the pipeline.


## Wrap-up

- `@ray.remote` + `.remote(...)` schedules; `ray.get` blocks and retrieves.
- Pass `ObjectRef`s as task arguments to build a graph; Ray waits on them for you.
- Actors hold mutable state and serialize calls per actor instance — use them when
  you need to share something (a model, a counter, a database handle) across tasks.

**Open the Ray Dashboard** (Anyscale Workspace → Ray Dashboard) and find the
`CatalogStats` actor and the tasks you just ran.